# Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
warnings.filterwarnings("ignore", ".*dubious year.*")
warnings.filterwarnings(
    "ignore", "Tried to get polar motions for times after IERS data is valid.*"
)

In [ ]:
import numpy as np
from astropy import units as u
from astropy.coordinates import (
    ICRS,
    EarthLocation,
    SkyCoord,
)
from astropy.time import Time
from astropy_healpix import HEALPix
from ligo.skymap import plot  # noqa: F401
from m4opt import fov
from m4opt.dynamics import nominal_roll
from m4opt.missions import uvex as mission
from m4opt.skygrid import _geodesic
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from regions import Regions
from tqdm.auto import tqdm

In [ ]:
(inscribed_fov,) = Regions.read("../fov-inscribed-circle.ds9")

Plot existing sky grid.

In [ ]:
i = np.arange(len(mission.skygrid))
fig = plt.figure(dpi=300)
ra0 = np.linspace(0, 360, 500, endpoint=False) * u.deg

with tqdm(total=len(ra0)) as progress:

    def animate(ra0):
        fig.clf()
        ax = plt.axes(projection="astro globe", center=SkyCoord(ra0, 30 * u.deg))
        plt.colorbar(
            ax.scatter_coord(
                mission.skygrid,
                c=i,
                s=4,
            ),
            orientation="vertical",
        ).set_label("Field ID")
        ax.coords["ra"].set_ticklabel(exclude_overlapping=True)
        ax.grid()
        progress.update()

    FuncAnimation(fig, animate, ra0, interval=50).save("../visualizations/skygrid.mp4")

Check that this is the coarsest grid that covers the entire sky with that circle.

In [ ]:
def skygrid_coverage_fraction(skygrid, region, nside=512):
    hpx = HEALPix(nside=nside, frame=ICRS())
    n_pix = hpx.npix
    n_covered = len(
        np.unique(np.concatenate(fov.footprint_healpix(hpx, region, skygrid)))
    )
    return n_covered / n_pix


n_lower = 4000
n_upper = 6000

best_n = np.inf

for b in tqdm(range(10, 50)):
    for c in range(0, b + 1):
        for base in ("icosahedron", "octahedron", "tetrahedron"):
            n = _geodesic.num_vertices(b, c, base)
            if n < n_lower or n > n_upper or n >= best_n:
                continue
            points = _geodesic.for_subdivision(b, c, base)
            if skygrid_coverage_fraction(points, inscribed_fov) >= 1:
                best_n = n
                best_b = b
                best_c = c
                best_base = base
                print(f"{n=} {base=} {b=} {c=}")

assert best_base == "icosahedron"
assert best_n == 5412
assert best_b == 21
assert best_c == 4

In [ ]:
def skygrid_multiple_coverage_fraction(time, region, nside=1024):
    hpx = HEALPix(nside=nside, frame=ICRS())
    n_pix = hpx.npix
    roll = nominal_roll(
        EarthLocation.from_geocentric(0 * u.m, 0 * u.m, 0 * u.m), mission.skygrid, time
    )
    _, counts = np.unique(
        np.concatenate(fov.footprint_healpix(hpx, region, mission.skygrid, roll)),
        return_counts=True,
    )
    return np.bincount(counts, minlength=10) / n_pix


t0 = Time("2025-01-01")
coverage_fraction = skygrid_multiple_coverage_fraction(t0, mission.fov)
coverage_fraction_inscribed = skygrid_multiple_coverage_fraction(t0, inscribed_fov)

width, height = plt.rcParams["figure.figsize"]
fig, axs = plt.subplots(1, 2, tight_layout=True, figsize=(2 * height, height))
values = coverage_fraction[1:4]
axs[0].pie(
    values,
    labels=[
        f"$n = {i + 1}$\n{np.around(100 * value):g}%" for i, value in enumerate(values)
    ],
    labeldistance=0.6,
    wedgeprops=dict(lw=1, ec="black"),
)
axs[0].set_title("Instantaneous FOV")
values = coverage_fraction_inscribed[1:3]
axs[1].pie(
    values,
    labels=[
        f"$n = {i + 1}$\n{np.around(100 * value):g}%" for i, value in enumerate(values)
    ],
    labeldistance=0.6,
    wedgeprops=dict(lw=1, ec="black"),
)
axs[1].set_title("Inscribed circle")
fig.savefig("../visualizations/coverage-fraction.pdf")

In [ ]:
center = SkyCoord("0d 0d")
radius = 10 * u.deg
width, _ = plt.rcParams["figure.figsize"]
fig = plt.figure(figsize=(width, width))
ax = fig.add_subplot(projection="astro zoom", center=center, radius=radius)
for key in ["ra", "dec"]:
    ax.coords[key].set_auto_axislabel(False)
    ax.coords[key].set_ticklabel_visible(False)
    ax.coords[key].set_ticks_visible(False)

t0 = Time("2025-01-01")
times = t0 + np.linspace(0, 1, 100, endpoint=False) * u.year
centers = mission.skygrid[
    mission.skygrid.separation(center) <= 1.1 * np.sqrt(2) * radius
]
geocenter = EarthLocation.from_geocentric(0 * u.m, 0 * u.m, 0 * u.m)

for region in fov.footprint(
    inscribed_fov,
    centers,
):
    ax.add_patch(
        region.to_pixel(ax.wcs).as_artist(
            edgecolor="black",
            alpha=0.5,
        )
    )

artists = []

with tqdm(total=len(times)) as progress:

    def animate(t):
        for artist in artists:
            artist.remove()
        del artists[:]
        rolls = nominal_roll(geocenter, centers, t)
        for region in fov.footprint(
            mission.fov,
            centers,
            rolls,
        ):
            artists.append(
                ax.add_patch(
                    region.to_pixel(ax.wcs).as_artist(
                        fill=True, facecolor="gray", edgecolor="black", alpha=0.25
                    )
                )
            )
        progress.update()

    FuncAnimation(fig, animate, times, interval=50).save(
        "../visualizations/skygrid-overlap.mp4"
    )